In [0]:
%sql
describe olist.bronze.order_items

In [0]:
%sql
--precios que no sean negativos y flete
Select * from olist.bronze.order_items where price < 0 or freight_value < 0



In [0]:
%sql
CREATE OR REPLACE TABLE olist.silver.order_items AS
SELECT 
    order_id,
    order_item_id,
    product_id,
    seller_id,
    shipping_limit_date,
    price,
    freight_value
from olist.bronze.order_items
where order_id is not null

In [0]:
%sql
SELECT
    order_id,
    order_item_id,
    COUNT(*) AS cantidad
FROM olist.silver.order_items
GROUP BY order_id, order_item_id
HAVING COUNT(*) > 1;

In [0]:
%sql 
describe olist.bronze.order_payments

In [0]:
%sql
--validacion de valores que no sean negativos
Select * from olist.bronze.order_payments where payment_value < 0 or payment_installments <= 0 or payment_sequential < 0

In [0]:
%sql
CREATE OR REPLACE TABLE olist.silver.order_payments AS 
SELECT
    order_id,
    payment_sequential,
    UPPER(TRIM(payment_type)) AS payment_type,
    payment_installments,
    payment_value
FROM olist.bronze.order_payments
where order_id is not null

In [0]:
%sql
SELECT
    order_id,
    payment_sequential,
    COUNT(*) AS cantidad
FROM olist.silver.order_payments
GROUP BY order_id, payment_sequential
HAVING COUNT(*) > 1;

In [0]:
%sql
describe olist.bronze.order_reviews

In [0]:
%sql
CREATE OR REPLACE TABLE olist.silver.order_reviews AS
SELECT
    review_id,
    order_id,
    review_score,
    review_comment_title,
    review_comment_message,
    review_creation_date,
    review_answer_timestamp
from olist.bronze.order_reviews
where review_id is not null
    

In [0]:
%sql
SELECT
    review_id,
    order_id,
    COUNT(*) AS cantidad
FROM olist.silver.order_reviews
GROUP BY review_id,order_id
HAVING COUNT(*) > 1;

In [0]:
%sql
describe olist.bronze.sellers

In [0]:
%sql
CREATE OR REPLACE TABLE olist.silver.sellers AS
SELECT 
    seller_id,
    cast(seller_zip_code_prefix as string) as seller_zip_code_prefix,
    UPPER(TRIM(seller_city)) as seller_city,
    UPPER(TRIM(seller_state)) as seller_state
FROM olist.bronze.sellers
where seller_id is not NULL

In [0]:
%sql
DESCRIBE olist.bronze.geolocation

In [0]:
%sql
CREATE OR REPLACE TABLE olist.silver.geolocation AS 
SELECT DISTINCT
    CAST(geolocation_zip_code_prefix as String) as geolocation_zip_code_prefix,
    geolocation_lat,
    geolocation_lng,
    UPPER(TRIM(geolocation_city)) as geolocation_city,
    UPPER(TRIM(geolocation_state)) as geolocation_state
FROM olist.bronze.geolocation
where geolocation_zip_code_prefix is not NULL

In [0]:
%sql
Select * from olist.silver.geolocation

In [0]:
%sql
Select 'Silver' as tabla ,count (*)as Total_filas from olist.silver.geolocation
union all 
select 'bronze' as tabla, count(*) as Total_filas from olist.bronze.geolocation 

In [0]:
%sql
describe olist.bronze.product_category_translation

In [0]:
%sql
CREATE OR REPLACE TABLE olist.silver.product_category_translation AS
SELECT DISTINCT
    UPPER(TRIM(product_category_name)) as product_category_name,
    upper(TRIM(product_category_name_english)) as product_category_name_english
FROM olist.bronze.product_category_translation
where product_category_name is not NULL


In [0]:
%sql
select * from olist.silver.product_category_translation

# Resumen - Silver Other Tables

En este notebook se construyeron y validaron las tablas Silver restantes necesarias para continuar con el análisis del proyecto.

## Order Items

Se creó la tabla `olist.silver.order_items` a partir de `olist.bronze.order_items`.

### Validaciones realizadas

- Se comprobó que `price` no contenga valores negativos.
- Se comprobó que `freight_value` no contenga valores negativos.
- Se validó que `shipping_limit_date` tenga tipo `TIMESTAMP`.
- Se verificó que la combinación `order_id + order_item_id` sea única.

### Transformaciones realizadas

- Se conservaron las claves `order_id`, `order_item_id`, `product_id` y `seller_id`.
- Se conservaron `shipping_limit_date`, `price` y `freight_value`.
- Se excluyeron registros con `order_id` nulo.

---

## Order Payments

Se creó la tabla `olist.silver.order_payments`.

### Validaciones realizadas

- Se verificó que `payment_value`, `payment_installments` y `payment_sequential` no tengan valores negativos.
- Se detectaron dos registros con `payment_installments = 0`.
- Estos registros fueron conservados, ya que no existe evidencia suficiente para clasificarlos como inválidos.

### Transformaciones realizadas

- Se normalizó `payment_type` eliminando espacios y convirtiendo el texto a mayúsculas.
- Se conservaron `payment_sequential`, `payment_installments` y `payment_value`.
- Se excluyeron registros con `order_id` nulo.

---

## Order Reviews

Se creó la tabla `olist.silver.order_reviews`.

### Validaciones realizadas

- Se confirmó que la ingesta multilínea corregida contiene 99.224 registros.
- Se verificó que `review_score` tenga valores válidos.
- No se detectaron duplicados después de corregir la ingesta.
- Se mantuvieron los campos de comentarios nulos, ya que son opcionales.

### Transformaciones realizadas

- Se conservaron identificadores, puntuación, comentarios y fechas.
- Se excluyeron registros con `review_id` nulo.

---

## Sellers

Se creó la tabla `olist.silver.sellers`.

### Transformaciones realizadas

- Se conservó `seller_id`.
- Se convirtió `seller_zip_code_prefix` a `STRING`.
- Se normalizaron `seller_city` y `seller_state` mediante `TRIM` y conversión a mayúsculas.
- Se excluyeron registros con `seller_id` nulo.

---

## Geolocation

Se creó la tabla `olist.silver.geolocation`.

### Validaciones realizadas

- En Bronze se identificaron 261.831 filas completamente duplicadas.
- Se comparó el total de registros antes y después de la limpieza:
  - Bronze: 1.000.163 registros.
  - Silver: 738.332 registros.
- La diferencia corresponde exactamente a los duplicados eliminados.

### Transformaciones realizadas

- Se eliminaron únicamente duplicados exactos mediante `DISTINCT`.
- Se convirtió `geolocation_zip_code_prefix` a `STRING`.
- Se normalizaron ciudad y estado.
- Se conservaron diferentes coordenadas asociadas a un mismo código postal.

---

## Product Category Translation

Se creó la tabla `olist.silver.product_category_translation`.

### Transformaciones realizadas

- Se conservaron `product_category_name` y `product_category_name_english`.
- Se eliminaron espacios innecesarios.
- Se normalizaron ambas categorías a mayúsculas.
- Se eliminaron duplicados exactos.
- Se excluyeron registros con `product_category_name` nulo.

## Resultado

Al finalizar este notebook, las tablas restantes fueron limpiadas, estandarizadas y almacenadas en la capa Silver.

Con esto, la capa Silver queda preparada para construir la capa Gold y comenzar a generar métricas y relaciones orientadas a responder la problemática del proyecto.